# 03: Feature engineering

**Owner:** John (Data Engineering & Acquisition Analyst) · **Ticket:** #5

Measures the engineered features that `capstone.features.add_features` adds to `anime.parquet`. These are the inputs for clustering, modeling and the final ranking. The notebook computes nothing new; it shows what each feature contains and where it falls short. The column definitions are in `docs/data-dictionary.md` ("Cleaned tables"), and the method write-up is in `report/data-cleaning.md`, section 7.

In [1]:
import pandas as pd

from capstone.dataset import load_anime
from capstone.features import PRIOR_REVIEWS

anime = load_anime()
print(f"{len(anime):,} anime x {anime.shape[1]} columns")

16,216 anime x 73 columns


## 1. Popularity index

`popularity_index` = the percentage of the catalog with **at most as many members** as this title (0–100). 100 is the most-watched title. A value of 90 means the title has at least as many members as 90% of the catalog.

It uses `members` only, so it doesn't overlap with `n_favorites`, which the clustering and models use as a separate loyalty input.

In [2]:
print(anime["popularity_index"].describe().round(2).to_string())
anime.nlargest(5, "members")[["title", "members", "popularity", "popularity_index"]]

count    16216.00
mean        50.02
std         28.85
min          0.01
25%         25.03
50%         50.01
75%         75.01
max        100.00


,title,members,popularity,popularity_index
1382,Death Note,1871043,1,100.000000
7402,Shingeki no Kyojin,1754979,2,99.993833
6573,Sword Art Online,1657823,3,99.987667
3946,Fullmetal Alchemist: Brotherhood,1615084,4,99.981500
10383,One Punch Man,1475219,5,99.975333


In [3]:
# What member count each index level corresponds to
levels = [50, 75, 90, 95, 99]
pd.DataFrame({
    "popularity_index at least": levels,
    "members at least": [int(anime.loc[anime["popularity_index"] >= p, "members"].min()) for p in levels],
})

,popularity_index at least,members at least
0,50,1839
1,75,11600
2,90,62378
3,95,137273
4,99,456083


## 2. Adjusted review score

Half of the reviewed anime have 3 reviews or fewer, so `mean_review_score` can swing on one review. `adjusted_review_score` blends each anime's reviews with **5 imaginary reviews at the catalog average**, the same idea as IMDb's weighted rating:

```
adjusted = (n_reviews × mean_review_score + 5 × catalog_mean) / (n_reviews + 5)
```

`catalog_mean` is the average over every review in the dataset. Titles with many reviews barely move, and titles with one or two are pulled strongly toward the average. Anime without reviews stay blank (no data, not a low score).

In [4]:
reviewed = anime[anime["n_reviews"] > 0]
catalog_mean = (reviewed["n_reviews"] * reviewed["mean_review_score"]).sum() / reviewed["n_reviews"].sum()
print(f"catalog mean review score: {catalog_mean:.2f}  (prior weight: {PRIOR_REVIEWS} reviews)")
shift = (reviewed["adjusted_review_score"] - reviewed["mean_review_score"]).abs()
print(f"median shift: {shift.median():.2f} points; titles with 20+ reviews: median shift {shift[reviewed['n_reviews'] >= 20].median():.3f}")

catalog mean review score: 7.33  (prior weight: 5 reviews)
median shift: 0.46 points; titles with 20+ reviews: median shift 0.064


In [5]:
# Perfect 10s from a single review: the raw mean ranks them above every classic; the adjusted score doesn't
cols = ["title", "n_reviews", "mean_review_score", "adjusted_review_score", "score"]
pd.concat([
    reviewed[reviewed["n_reviews"] == 1].nlargest(3, "mean_review_score")[cols],
    reviewed.nlargest(3, "members")[cols],
])

,title,n_reviews,mean_review_score,adjusted_review_score,score
395,Shiritsu Araiso Koutougakkou Seitokai Shikkoubu,1,10.000000,7.774805,6.51
532,Moldiver,1,10.000000,7.774805,6.18
589,Mahoromatic Summer Special,1,10.000000,7.774805,7.18
1382,Death Note,854,8.809133,8.800522,8.65
7402,Shingeki no Kyojin,20,8.250000,8.065953,8.47
6573,Sword Art Online,1292,6.390093,6.393715,7.49


## 3. Genre features

- `genre_count`: the number of genre tags.
- One true/false `genre_<name>` column per genre (43 in all), e.g. `genre_action`, `genre_slice_of_life`, `genre_sci_fi`. Leonardo can use these to compare genres, and the models and clustering can take them as inputs directly.

In [6]:
flag_cols = [c for c in anime.columns if c.startswith("genre_") and c != "genre_count"]
print(f"{len(flag_cols)} genre flag columns; genre_count ranges {anime['genre_count'].min()}-{anime['genre_count'].max()}, mean {anime['genre_count'].mean():.1f}")
anime[flag_cols].sum().sort_values(ascending=False).rename("anime").head(10).to_frame()

43 genre flag columns; genre_count ranges 0-13, mean 2.9


,anime
genre_comedy,5640
genre_action,3572
genre_fantasy,2971
genre_adventure,2753
genre_sci_fi,2462
genre_kids,2458
genre_drama,2457
genre_shounen,1915
genre_music,1863
genre_romance,1762


In [7]:
# The flags must agree with the genre lists they came from
assert (anime[flag_cols].sum(axis=1) == anime["genre_count"]).all()
print("PASS  every genre flag matches the genre list")

PASS  every genre flag matches the genre list


## 4. Franchise features

The dataset has no "related titles" field, but MyAnimeList lists every season, movie and spin-off as its own entry. Franchises are therefore inferred from titles. `franchise_root` reduces a title to its root by removing:

- a subtitle after `": "` ("Naruto: Shippuuden" → "naruto")
- an edition in brackets ("Hunter x Hunter (2011)", "Detective Conan (TV)")
- trailing sequel markers: "Season 2", "2nd Season", "Second Season", "Part 2", "S2", "II", "R2", a trailing number, "Movie", "OVA", "Specials", "TV"

Titles with the same root form one franchise:

| Column | Meaning |
|---|---|
| `franchise` | the root |
| `franchise_size` | number of entries (seasons, movies, OVAs, specials) |
| `franchise_members` | members summed across the entries. The same viewer is counted once per entry they listed, so this is total reach, not unique people. |
| `is_sequel` | True unless this is the franchise's earliest-starting entry. Undated entries count as later; ties go to the lowest `uid`. |

In [8]:
multi = anime[anime["franchise_size"] > 1]
print(f"{len(multi):,} anime ({len(multi) / len(anime):.0%}) belong to {multi['franchise'].nunique():,} franchises with 2+ entries")
print(f"{anime['is_sequel'].sum():,} entries are marked as sequels / side stories")
anime.nsmallest(20, "popularity")[["title", "franchise", "franchise_size", "is_sequel"]]

6,277 anime (39%) belong to 1,983 franchises with 2+ entries
4,294 entries are marked as sequels / side stories


,title,franchise,franchise_size,is_sequel
1382,Death Note,death note,2,False
7402,Shingeki no Kyojin,shingeki no kyojin,12,False
6573,Sword Art Online,sword art online,14,False
3946,Fullmetal Alchemist: Brotherhood,fullmetal alchemist,9,True
10383,One Punch Man,one punch man,5,False
8589,Tokyo Ghoul,tokyo ghoul,3,False
5645,Steins;Gate,steins;gate,7,False
8098,No Game No Life,no game no life,4,False
10,Naruto,naruto,18,False
4609,Angel Beats!,angel beats!,3,False


In [9]:
# Largest franchises by total reach: candidates for the 3 priority franchise investments
anime.drop_duplicates("franchise").nlargest(10, "franchise_members")[["franchise", "franchise_size", "franchise_members"]]

,franchise,franchise_size,franchise_members
7402,shingeki no kyojin,12,4580461
6573,sword art online,14,4174729
10,naruto,18,3985026
11113,boku no hero academia,6,3585901
1420,code geass,29,3246089
100,fullmetal alchemist,9,3198589
5645,steins;gate,7,2640147
9668,shokugeki no souma,9,2531808
10383,one punch man,5,2479338
329,fate/stay night,9,2356386


### Known limits

Title matching is a heuristic. On the most popular titles it groups correctly (above), but it fails in three known ways. Anyone citing a franchise figure should check the entries behind it.

In [10]:
def entries(root):
    return anime.loc[anime["franchise"] == root, "title"].tolist()

print("1. Unrelated titles sharing a root:")
print("   sin:", entries("sin"))
print("2. One franchise split across roots:")
for root in ["k-on!", "k-on!!", "naruto", "boruto"]:
    print(f"   {root}: {len(entries(root))} entries")
print("3. Remakes and side stories count as sequels:")
print(anime.loc[anime["title"].isin(["Fullmetal Alchemist", "Fullmetal Alchemist: Brotherhood"]), ["title", "start_date", "is_sequel"]].to_string(index=False))

1. Unrelated titles sharing a root:
   sin: ['Sin: The Movie', 'Sin: Nanatsu no Taizai', 'Sin: Nanatsu no Taizai Zange-roku', 'Sin: Nanatsu no Taizai Zange-roku Specials', 'Sin: Nanatsu no Taizai - Masani Akuma no Shogyou...']
2. One franchise split across roots:
   k-on!: 4 entries
   k-on!!: 3 entries
   naruto: 18 entries
   boruto: 4 entries
3. Remakes and side stories count as sequels:
                           title start_date  is_sequel
             Fullmetal Alchemist 2003-10-04      False
Fullmetal Alchemist: Brotherhood 2009-04-05       True


## 5. Final table

In [11]:
new = ["popularity_index", "adjusted_review_score", "genre_count", "franchise", "franchise_size", "franchise_members", "is_sequel"]
print(f"anime.parquet: {len(anime):,} rows x {anime.shape[1]} columns ({len(new)} features + {len(flag_cols)} genre flags added)")
anime[new].dtypes.astype(str).rename("dtype").to_frame()

anime.parquet: 16,216 rows x 73 columns (7 features + 43 genre flags added)


,dtype
popularity_index,float64
adjusted_review_score,float64
genre_count,int64
franchise,str
franchise_size,int64
franchise_members,int64
is_sequel,bool
